In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from ast import literal_eval
import numpy as np

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('./data/movies_metadata.csv', low_memory=False)

# Reducir tamaño para evitar consumo excesivo de RAM
df = df.head(5000)

df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [4]:
C = df['vote_average'].mean()

print(C)

6.06916


In [5]:
m = df['vote_count'].quantile(0.90)

print(m)

568.1000000000004


In [6]:
q_movies = df.copy().loc[df['vote_count'] >= m]

print(q_movies.shape)

(500, 24)


In [7]:
def weighted_rating(x, m=m, C=C):

    v = x['vote_count']
    R = x['vote_average']

    return (v / (v + m) * R) + (m / (m + v) * C)

In [8]:
q_movies['score'] = q_movies.apply(weighted_rating, axis=1)

In [9]:
q_movies = q_movies.sort_values('score', ascending=False)

q_movies[['title', 'vote_average', 'vote_count', 'score']].head(15)

,title,vote_average,vote_count,score
314,The Shawshank Redemption,8.5,8358.0,8.345290
834,The Godfather,8.5,6024.0,8.290513
2843,Fight Club,8.3,9678.0,8.176310
292,Pulp Fiction,8.3,8670.0,8.162814
351,Forrest Gump,8.2,8147.0,8.061100
522,Schindler's List,8.3,4436.0,8.046740
1154,The Empire Strikes Back,8.2,5998.0,8.015639
2211,Life Is Beautiful,8.3,3643.0,7.999048
1178,The Godfather: Part II,8.3,3418.0,7.982060
289,Leon: The Professional,8.2,4293.0,7.950976


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [11]:
df['overview'] = df['overview'].fillna('')

In [12]:
tfidf = TfidfVectorizer(stop_words='english')

In [13]:
tfidf_matrix = tfidf.fit_transform(df['overview'])

print(tfidf_matrix.shape)

(5000, 22304)


In [14]:
from sklearn.metrics.pairwise import linear_kernel

cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

print(cosine_sim.shape)

(5000, 5000)


In [15]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

In [16]:
def get_recommendations(title, cosine_sim=cosine_sim):

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Ignorar la misma película
    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]

    return df['title'].iloc[movie_indices]